# Clean, color, cue-conflict, translation and patch evaluation
Reorganized only; no experiment cells executed during creation. Existing results were copied with byte-hash verification.

In [ ]:
def compute_metrics(logits, labels):
    probabilities = logits.softmax(dim=1)
    confidence, predictions = probabilities.max(dim=1)

    y_true = labels.numpy()
    y_pred = predictions.numpy()

    metrics = {
        "top1_accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(num_classes)),
            average="macro",
            zero_division=0,
        ),
        "mean_max_confidence": confidence.mean().item(),
    }

    return metrics, predictions, confidence, probabilities




In [ ]:
def run_clean():
    display(pd.read_csv(TASK_DIR / 'results/clean_metrics.csv'))
    print('Saved baseline reused; no head training or feature extraction.')
    return baseline



In [ ]:
def run_color():
    color_images, color_loaders = make_color_images()
    def show_color_examples(number=5, seed=SEED):
        rng = np.random.default_rng(seed)

        # Choose different classes so the examples are varied.
        selected_classes = rng.choice(
            num_classes,
            size=min(number, num_classes),
            replace=False,
        )

        positions = [
            int(rng.choice(
                np.flatnonzero(
                    test_manifest["label"].to_numpy() == class_id
                )
            ))
            for class_id in selected_classes
        ]

        fig, axes = plt.subplots(
            len(positions),
            3,
            figsize=(10, 3 * len(positions)),
            squeeze=False,
        )

        for row, position in enumerate(positions):
            entry = test_manifest.iloc[position]
            image, _ = official_test[int(entry["official_test_index"])]
            original = common_preprocess(image.convert("RGB"))

            show_tensor(
                axes[row, 0],
                original,
                f"Original: {entry['class_name']}",
            )
            show_tensor(
                axes[row, 1],
                color_images["grayscale"][position],
                "Grayscale",
            )
            show_tensor(
                axes[row, 2],
                color_images["hue_rotation"][position],
                "Hue rotation: +90°",
            )

        fig.suptitle("Color interventions", fontsize=15)
        plt.tight_layout()
        plt.show()


    show_color_examples(number=5)
    color_logits = {}
    color_feature_bank = {}

    for name in ["resnet50", "vit_b16", "clip"]:
        backbone, normalize = load_backbone(name)
        color_feature_bank[name] = {}

        # Prepare the same fixed text prompts used in the clean baseline.
        if name == "clip":
            prompts = [
                f"a photo of a {class_name}."
                for class_name in class_names
            ]

            tokenizer = open_clip.get_tokenizer("ViT-B-32")
            tokens = tokenizer(prompts).to(DEVICE)

            with torch.no_grad():
                text_features = backbone.encode_text(tokens)
                text_features = F.normalize(
                    text_features.float(), dim=-1
                ).cpu()

                similarity_scale = (
                    backbone.logit_scale.exp().float().cpu()
                )

        for condition, loader in color_loaders.items():
            extracted = extract_features(
                backbone,
                normalize,
                loader,
                name,
                condition,
            )

            assert torch.equal(extracted["labels"], y_test)

            features = extracted["features"]
            color_feature_bank[name][condition] = features

            # Evaluate the existing clean-trained head without retraining.
            heads[name].eval()

            with torch.no_grad():
                color_logits[(display_names[name], condition)] = (
                    heads[name](features)
                )

                if name == "clip":
                    color_logits[(
                        "CLIP ViT-B/32 zero-shot", condition
                    )] = (
                        similarity_scale
                        * (features @ text_features.T)
                    )

        del backbone, normalize
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    color_metric_rows = []
    color_prediction_tables = []

    for (model_name, condition), logits in color_logits.items():
        metrics, predictions, confidence, _ = compute_metrics(
            logits, y_test
        )

        clean_predictions_for_model = (
            clean_logits[model_name].argmax(dim=1)
        )

        clean_accuracy = (
            (clean_predictions_for_model == y_test)
            .float()
            .mean()
            .item()
        )

        consistency = (
            (predictions == clean_predictions_for_model)
            .float()
            .mean()
            .item()
        )

        color_metric_rows.append({
            "model": model_name,
            "condition": condition,
            **metrics,
            "clean_accuracy": clean_accuracy,
            "accuracy_change_pp": 100 * (
                metrics["top1_accuracy"] - clean_accuracy
            ),
            "prediction_consistency": consistency,
        })

        prediction_table = test_manifest.copy()
        prediction_table["model"] = model_name
        prediction_table["condition"] = condition
        prediction_table["predicted_label"] = predictions.numpy()
        prediction_table["predicted_class"] = [
            class_names[index] for index in predictions.tolist()
        ]
        prediction_table["confidence"] = confidence.numpy()
        prediction_table["clean_predicted_label"] = (
            clean_predictions_for_model.numpy()
        )
        prediction_table["matches_clean"] = (
            predictions == clean_predictions_for_model
        ).numpy()

        color_prediction_tables.append(prediction_table)

    color_results = pd.DataFrame(color_metric_rows)

    color_predictions = pd.concat(
        color_prediction_tables,
        ignore_index=True,
    )

    display(
        color_results[
            [
                "model",
                "condition",
                "clean_accuracy",
                "top1_accuracy",
                "accuracy_change_pp",
                "prediction_consistency",
                "macro_f1",
                "mean_max_confidence",
            ]
        ].style.format({
            "clean_accuracy": "{:.2%}",
            "top1_accuracy": "{:.2%}",
            "accuracy_change_pp": "{:+.2f}",
            "prediction_consistency": "{:.2%}",
            "macro_f1": "{:.4f}",
            "mean_max_confidence": "{:.2%}",
        })
    )
    color_results.to_csv(TASK_DIR/'results/color_metrics.csv',index=False)
    color_predictions.to_csv(TASK_DIR/'results/color_predictions.csv',index=False)
    torch.save(dict(features=color_feature_bank,logits=color_logits,test_indices=torch.as_tensor(test_indices)),
               TASK_DIR/'results/color_evaluation.pt')



In [ ]:
def run_cue_conflicts():
    package, conflict_manifest, review_log = load_preserved_conflicts()
    conflict_images = package['images']
    content_targets, style_targets = package['content_targets'], package['style_targets']
    # Cached outputs belong to the preserved baseline copied unchanged at creation.
    provenance = json.loads((TASK_DIR / 'configs/preservation.json').read_text(encoding='utf-8'))
    matches = [
        record for record in provenance['imported_results']
        if record['copy'].replace(chr(92), '/') == 'task1/results/clean_baseline.pt'
    ]
    if len(matches) != 1:
        raise RuntimeError(
            'Expected exactly one clean-baseline provenance record in preservation.json. '
            'Preserved cue images are unchanged; do not regenerate them.')
    expected = matches[0]['sha256']
    if file_sha256(BASELINE_PATH) != expected:
        raise RuntimeError('Baseline changed: do not mix cached cue predictions with different heads.')
    conflict_logits = package['logits']
    conflict_feature_bank = package['features']
    # CELL 13
    conflict_prediction_tables = []

    for model_name, logits in conflict_logits.items():
        predictions = logits.argmax(dim=1)

        table = conflict_manifest.copy()
        table["model"] = model_name
        table["predicted_label"] = predictions.numpy()
        table["predicted_class"] = [
            class_names[index] for index in predictions.tolist()
        ]

        table["decision"] = np.where(
            predictions.numpy() == content_targets.numpy(),
            "shape",
            np.where(
                predictions.numpy() == style_targets.numpy(),
                "texture",
                "other",
            ),
        )

        conflict_prediction_tables.append(table)

    conflict_predictions = pd.concat(
        conflict_prediction_tables,
        ignore_index=True,
    )


    def summarize_conflicts(frame, group_columns):
        rows = []

        for keys, group in frame.groupby(group_columns, sort=False):
            if not isinstance(keys, tuple):
                keys = (keys,)

            shape_count = int((group["decision"] == "shape").sum())
            texture_count = int((group["decision"] == "texture").sum())
            other_count = int((group["decision"] == "other").sum())

            covered = shape_count + texture_count
            total = len(group)

            rows.append({
                **dict(zip(group_columns, keys)),
                "Nshape": shape_count,
                "Ntexture": texture_count,
                "Nother": other_count,
                "Ntotal": total,
                "shape_bias_pct": (
                    100 * shape_count / covered
                    if covered > 0 else np.nan
                ),
                "coverage_pct": 100 * covered / total,
            })

        return pd.DataFrame(rows)


    shape_texture_results = summarize_conflicts(
        conflict_predictions, ["model"]
    )

    shape_texture_by_direction = summarize_conflicts(
        conflict_predictions, ["model", "direction"]
    )

    display(
        shape_texture_results.style.format({
            "shape_bias_pct": "{:.2f}",
            "coverage_pct": "{:.2f}",
        }, na_rep="Undefined")
    )

    display(
        shape_texture_by_direction.style.format({
            "shape_bias_pct": "{:.2f}",
            "coverage_pct": "{:.2f}",
        }, na_rep="Undefined")
    )
    shape_texture_results.to_csv(TASK_DIR/'results/shape_texture_metrics.csv',index=False)
    shape_texture_by_direction.to_csv(TASK_DIR/'results/shape_texture_by_direction.csv',index=False)
    conflict_predictions.to_csv(TASK_DIR/'results/cue_conflict_predictions.csv',index=False)
    # Show preserved images and their predictions; no generation or selection.
    fig,axes=plt.subplots(2,3,figsize=(14,8))
    for i,ax in enumerate(axes.flat):
        ax.imshow(conflict_images[i].permute(1,2,0).numpy())
        guesses='; '.join(f"{name}: {class_names[int(logits[i].argmax())]}" for name,logits in conflict_logits.items())
        ax.set_title(conflict_manifest.iloc[i]['direction']+'\n'+guesses,fontsize=7,wrap=True)
        ax.axis('off')
    fig.tight_layout()
    fig.savefig(TASK_DIR/'results/cue_conflict_examples.png',dpi=160)
    plt.show()



In [ ]:
def run_translation():
    # CELL 02 — Imports, paths and reproducibility
    from pathlib import Path
    import gc
    import random
    import hashlib
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import matplotlib.pyplot as plt
    from torchvision import datasets, models, transforms
    from torchvision.transforms import InterpolationMode
    from torch.utils.data import Dataset, DataLoader
    from tqdm.auto import tqdm
    from IPython.display import display
    import open_clip

    DATA_DIR = TASK_DIR / 'data'
    BASELINE_PATH = TASK_DIR / 'results' / 'clean_baseline.pt'
    OUTPUT_DIR = TASK_DIR / 'results' / 'translation'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    SEED = 6304
    BATCH_SIZE = 32
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    print('Device:', DEVICE)
    print('Results:', OUTPUT_DIR)

    # CELL 03 — Load saved heads, clean predictions and identical test images
    baseline = torch.load(BASELINE_PATH, map_location='cpu', weights_only=True)
    assert baseline['seed'] == SEED
    class_names = list(baseline['class_names'])
    num_classes = len(class_names)
    test_indices = baseline['test_indices'].long().numpy()
    official_test = datasets.STL10(str(DATA_DIR), split='test', download=False)
    assert list(official_test.classes) == class_names
    y_test = torch.tensor(np.asarray(official_test.labels)[test_indices], dtype=torch.long)
    assert len(test_indices) == 500 and len(np.unique(test_indices)) == 500

    common_preprocess = transforms.Compose([
        transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.ToTensor(),
    ])
    # Approximately 300 MB, shared across all models. No transformed copies retained.
    clean_images = torch.stack([
        common_preprocess(official_test[int(i)][0].convert('RGB')) for i in test_indices
    ])
    heads = {}
    for name, state in baseline['head_states'].items():
        out_dim, in_dim = state['weight'].shape
        head = nn.Linear(in_dim, out_dim)
        head.load_state_dict(state)
        heads[name] = head.eval().requires_grad_(False)
    DISPLAY_NAMES = {
        'resnet50': 'ResNet-50 + linear head',
        'vit_b16': 'ViT-B/16 + linear head',
        'clip': 'CLIP ViT-B/32 + linear head',
    }
    ZERO_SHOT_NAME = 'CLIP ViT-B/32 zero-shot'
    clean_logits = baseline['clean_logits']
    clean_predictions = {name: logits.argmax(1) for name, logits in clean_logits.items()}
    for logits in clean_logits.values(): assert logits.shape == (500, num_classes)
    manifest = pd.DataFrame({'official_test_index': test_indices, 'label': y_test.numpy()})
    manifest['image_id'] = [f'stl10/test/{i:05d}' for i in test_indices]
    manifest['class_name'] = [class_names[int(i)] for i in y_test]
    print('Loaded 500 images and all trained heads. No training required.')

    CONDITIONS = [(d, direction) for d in DISPLACEMENTS if d for direction in DIRECTIONS]
    # CELL 05 — PREVIEW before evaluation — edit SAMPLE_POSITION here
    SAMPLE_POSITION = 0  # 0–499; this is a position in the fixed subset.
    image = clean_images[SAMPLE_POSITION]
    label = class_names[int(y_test[SAMPLE_POSITION])]
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for row, (direction, (ux, uy)) in enumerate(DIRECTIONS.items()):
        for column, displacement in enumerate(DISPLACEMENTS):
            shown = translate_rgb(image, displacement * ux, displacement * uy)
            axes[row, column].imshow(shown.permute(1, 2, 0).numpy())
            axes[row, column].set_title(f'{direction}: {displacement} px')
            axes[row, column].axis('off')
    fig.suptitle(f'{label} | official test ID {test_indices[SAMPLE_POSITION]}')
    plt.tight_layout()
    plt.show()

    # CELL 07 — Inference helpers and automatic resume validation
    @torch.no_grad()
    def encode_loader(backbone, normalize, loader, name):
        parts = []
        for images in tqdm(loader, desc=f'{name} batches', leave=False):
            images = normalize(images.to(DEVICE))
            if name == 'clip':
                features = F.normalize(backbone.encode_image(images).float(), dim=-1)
            else:
                features = backbone(images).float()
            parts.append(features.cpu())
        return torch.cat(parts)

    @torch.no_grad()
    def clip_text_setup(backbone):
        tokens = open_clip.get_tokenizer('ViT-B-32')([
            f'a photo of a {name}.' for name in class_names
        ]).to(DEVICE)
        text = F.normalize(backbone.encode_text(tokens).float(), dim=-1).cpu()
        return text, backbone.logit_scale.exp().float().cpu()

    # Detect changed baseline or transform code before reusing partial results.
    protocol = 'translation-v1;RGB-bicubic224;reflect-pad=max-shift;crop=p-dy,p-dx;d=8,16,32'
    baseline_hash = hashlib.sha256(BASELINE_PATH.read_bytes()).hexdigest()

    def cache_path(name, displacement, direction):
        return OUTPUT_DIR / f'{name}_{displacement:02d}_{direction}.pt'

    def read_cached(path):
        package = torch.load(path, map_location='cpu', weights_only=True)
        if package['protocol'] != protocol or package['baseline_sha256'] != baseline_hash:
            raise RuntimeError(f'Checkpoint does not match this experiment: {path}')
        assert torch.equal(package['test_indices'], torch.as_tensor(test_indices))
        assert package['features'].shape[0] == len(test_indices)
        return package

    def atomic_save(package, path):
        temporary = path.with_suffix('.tmp')
        torch.save(package, temporary)
        temporary.replace(path)

    print('Evaluation saves each completed condition and resumes on rerun.')

    # CELL 08 — EVALUATE — slow cell; progress saved after every condition
    for name in DISPLAY_NAMES:
        remaining = []
        for displacement, direction in CONDITIONS:
            path = cache_path(name, displacement, direction)
            if path.exists():
                read_cached(path)
            else:
                remaining.append((displacement, direction))
        if not remaining:
            print(name, 'already complete; reusing saved results.')
            continue
        backbone, normalize = load_backbone(name)
        if name == 'clip': text_features, logit_scale = clip_text_setup(backbone)
        for displacement, direction in tqdm(remaining, desc=name):
            loader = DataLoader(TranslationImages(clean_images, displacement, direction),
                                batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
            features = encode_loader(backbone, normalize, loader, name)
            with torch.no_grad():
                logits = {DISPLAY_NAMES[name]: heads[name](features)}
                if name == 'clip':
                    logits[ZERO_SHOT_NAME] = logit_scale * (features @ text_features.T)
            atomic_save({
                'protocol': protocol, 'baseline_sha256': baseline_hash,
                'test_indices': torch.as_tensor(test_indices),
                'backbone': name, 'displacement': displacement, 'direction': direction,
                'features': features, 'logits': logits,
            }, cache_path(name, displacement, direction))
        del backbone, normalize
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('All translated conditions evaluated. Run cells 09–11.')

    # CELL 09 — Metrics — direction results and averages
    metric_rows, prediction_tables = [], []
    def record_metrics(model_name, displacement, direction, logits):
        probabilities = logits.softmax(1)
        confidence, predicted = probabilities.max(1)
        clean = clean_predictions[model_name]
        accuracy = (predicted == y_test).float().mean().item()
        clean_accuracy = (clean == y_test).float().mean().item()
        consistency = (predicted == clean).float().mean().item()
        metric_rows.append(dict(model=model_name, displacement=displacement, direction=direction,
            accuracy=accuracy, prediction_consistency=consistency,
            accuracy_change_pp=100*(accuracy-clean_accuracy)))
        table = manifest.copy()
        table['model'], table['displacement'], table['direction'] = model_name, displacement, direction
        table['predicted_label'] = predicted.numpy()
        table['clean_predicted_label'] = clean.numpy()
        table['matches_clean'] = (predicted == clean).numpy()
        table['confidence'] = confidence.numpy()
        prediction_tables.append(table)
    for model_name, logits in clean_logits.items():
        record_metrics(model_name, 0, 'clean', logits)
    for name in DISPLAY_NAMES:
        for displacement, direction in CONDITIONS:
            path = cache_path(name, displacement, direction)
            if not path.exists(): raise RuntimeError('Finish cell 08 first. Missing: ' + path.name)
            package = read_cached(path)
            for model_name, logits in package['logits'].items():
                record_metrics(model_name, displacement, direction, logits)
    translation_by_direction = pd.DataFrame(metric_rows)
    translation_summary = translation_by_direction.groupby(['model','displacement'], as_index=False)[
        ['accuracy','prediction_consistency','accuracy_change_pp']
    ].mean()
    translation_predictions = pd.concat(prediction_tables, ignore_index=True)
    assert len(translation_summary) == 16
    assert len(translation_predictions) == 4 * 13 * len(test_indices)
    display(translation_summary.style.format({
        'accuracy':'{:.2%}', 'prediction_consistency':'{:.2%}', 'accuracy_change_pp':'{:+.2f}'
    }))

    # CELL 10 — Plot accuracy and prediction consistency
    translation_figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    for model_name, group in translation_summary.groupby('model', sort=False):
        group = group.sort_values('displacement')
        axes[0].plot(group['displacement'], 100*group['accuracy'], marker='o', label=model_name)
        axes[1].plot(group['displacement'], 100*group['prediction_consistency'], marker='o', label=model_name)
    for ax, title in zip(axes, ['Top-1 accuracy (%)', 'Prediction consistency (%)']):
        ax.set_xlabel('Displacement (pixels)')
        ax.set_ylabel(title)
        ax.set_xticks(DISPLACEMENTS)
        ax.set_ylim(0, 102)
        ax.grid(alpha=0.25)
    axes[1].legend(fontsize=8)
    translation_figure.suptitle('Translation: mean over four directions; zero is the clean baseline')
    plt.tight_layout()
    plt.show()

    # CELL 11 — Save tables and the plot
    translation_summary.to_csv(OUTPUT_DIR / 'translation_summary.csv', index=False)
    translation_by_direction.to_csv(OUTPUT_DIR / 'translation_by_direction.csv', index=False)
    translation_predictions.to_csv(OUTPUT_DIR / 'translation_predictions.csv', index=False)
    manifest.to_csv(OUTPUT_DIR / 'test_manifest.csv', index=False)
    translation_figure.savefig(OUTPUT_DIR / 'translation_curves.png', dpi=180, bbox_inches='tight')
    print('Saved tables and plot to:', OUTPUT_DIR)
    print('Per-condition .pt files already contain features and logits for representation analysis.')



In [ ]:
def run_patch_shuffle():
    # CELL 02 — Imports, paths and reproducibility
    from pathlib import Path
    import gc
    import random
    import hashlib
    import numpy as np
    import pandas as pd
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import matplotlib.pyplot as plt
    from torchvision import datasets, models, transforms
    from torchvision.transforms import InterpolationMode
    from torch.utils.data import Dataset, DataLoader
    from tqdm.auto import tqdm
    from IPython.display import display
    import open_clip

    DATA_DIR = TASK_DIR / 'data'
    BASELINE_PATH = TASK_DIR / 'results' / 'clean_baseline.pt'
    OUTPUT_DIR = TASK_DIR / 'results' / 'patch_shuffle'
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    SEED = 6304
    BATCH_SIZE = 32
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    print('Device:', DEVICE)
    print('Results:', OUTPUT_DIR)

    from sklearn.metrics import f1_score

    # CELL 03 — Load saved baseline and the fixed test subset
    baseline = torch.load(BASELINE_PATH, map_location='cpu', weights_only=True)
    assert baseline['seed'] == SEED
    class_names = list(baseline['class_names'])
    num_classes = len(class_names)
    test_indices = baseline['test_indices'].long().numpy()
    official_test = datasets.STL10(str(DATA_DIR), split='test', download=False)
    assert list(official_test.classes) == class_names
    y_test = torch.tensor(np.asarray(official_test.labels)[test_indices], dtype=torch.long)
    assert len(test_indices) == 500 and len(np.unique(test_indices)) == 500

    common_preprocess = transforms.Compose([
        transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC, antialias=True),
        transforms.ToTensor(),
    ])
    # Approximately 300 MB, shared across all models. No transformed copies retained.
    clean_images = torch.stack([
        common_preprocess(official_test[int(i)][0].convert('RGB')) for i in test_indices
    ])
    heads = {}
    for name, state in baseline['head_states'].items():
        out_dim, in_dim = state['weight'].shape
        head = nn.Linear(in_dim, out_dim)
        head.load_state_dict(state)
        heads[name] = head.eval().requires_grad_(False)
    DISPLAY_NAMES = {
        'resnet50': 'ResNet-50 + linear head',
        'vit_b16': 'ViT-B/16 + linear head',
        'clip': 'CLIP ViT-B/32 + linear head',
    }
    ZERO_SHOT_NAME = 'CLIP ViT-B/32 zero-shot'
    clean_logits = baseline['clean_logits']
    clean_predictions = {name: logits.argmax(1) for name, logits in clean_logits.items()}
    for logits in clean_logits.values(): assert logits.shape == (500, num_classes)
    manifest = pd.DataFrame({'official_test_index': test_indices, 'label': y_test.numpy()})
    manifest['image_id'] = [f'stl10/test/{i:05d}' for i in test_indices]
    manifest['class_name'] = [class_names[int(i)] for i in y_test]
    print('Loaded 500 images and all trained heads. No training required.')

    permutations, shuffled_images = make_patch_images(clean_images, test_indices)
    # CELL 05 — PREVIEW original and shuffled images before evaluation
    PREVIEW_POSITIONS = [0, 1, 2, 3, 4]  # Positions in the saved subset, from 0 to 499.
    fig, axes = plt.subplots(len(PREVIEW_POSITIONS), 2,
                             figsize=(7, 3 * len(PREVIEW_POSITIONS)), squeeze=False)
    for row, position in enumerate(PREVIEW_POSITIONS):
        label = class_names[int(y_test[position])]
        for column, images in enumerate([clean_images, shuffled_images]):
            axes[row, column].imshow(images[position].permute(1, 2, 0).numpy())
            condition = 'Original' if column == 0 else 'Shuffled 4 x 4'
            axes[row, column].set_title(f'{condition}: {label} | ID {test_indices[position]}')
            axes[row, column].axis('off')
    plt.tight_layout()
    plt.show()

    # CELL 07 — Inference and saved-result validation
    @torch.no_grad()
    def encode_images(backbone, normalize, images, name):
        parts = []
        loader = DataLoader(images, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
        for batch in tqdm(loader, desc=name):
            batch = normalize(batch.to(DEVICE))
            if name == 'clip': features = F.normalize(backbone.encode_image(batch).float(), dim=-1)
            else: features = backbone(batch).float()
            parts.append(features.cpu())
        return torch.cat(parts)

    @torch.no_grad()
    def clip_text_setup(backbone):
        tokens = open_clip.get_tokenizer('ViT-B-32')([
            f'a photo of a {name}.' for name in class_names
        ]).to(DEVICE)
        return (F.normalize(backbone.encode_text(tokens).float(), dim=-1).cpu(),
                backbone.logit_scale.exp().float().cpu())

    PROTOCOL = 'patch-v1;RGB-bicubic224;4x4;56px;output-to-input-permutation'
    baseline_hash = hashlib.sha256(BASELINE_PATH.read_bytes()).hexdigest()

    def atomic_save(package, path):
        temporary = path.with_suffix('.tmp')
        torch.save(package, temporary)
        temporary.replace(path)

    def load_result(name):
        package = torch.load(OUTPUT_DIR / f'{name}.pt', map_location='cpu', weights_only=True)
        assert package['protocol'] == PROTOCOL, 'Saved protocol differs.'
        assert package['baseline_sha256'] == baseline_hash, 'Saved baseline differs.'
        assert torch.equal(package['test_indices'], torch.as_tensor(test_indices))
        assert torch.equal(package['permutations'], permutations), 'Saved permutations differ.'
        assert package['features'].shape[0] == len(test_indices)
        return package
    print('Helpers ready. Cell 08 saves after each backbone and skips matching completed results.')

    # CELL 08 — EVALUATE all four classifiers — no retraining
    # Save the exact transform definition before evaluation.
    atomic_save({'protocol': PROTOCOL, 'seed': SEED,
                 'test_indices': torch.as_tensor(test_indices), 'permutations': permutations},
                OUTPUT_DIR / 'patch_permutations.pt')
    for name in DISPLAY_NAMES:
        path = OUTPUT_DIR / f'{name}.pt'
        if path.exists():
            load_result(name)
            print(name, 'already complete; using saved results.')
            continue
        backbone, normalize = load_backbone(name)
        features = encode_images(backbone, normalize, shuffled_images, name)
        with torch.no_grad():
            logits = {DISPLAY_NAMES[name]: heads[name](features)}
            if name == 'clip':
                text_features, scale = clip_text_setup(backbone)
                logits[ZERO_SHOT_NAME] = scale * (features @ text_features.T)
        atomic_save({'protocol': PROTOCOL, 'baseline_sha256': baseline_hash,
                     'test_indices': torch.as_tensor(test_indices), 'permutations': permutations,
                     'features': features, 'logits': logits}, path)
        del backbone, normalize
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('Evaluation complete. Continue with cells 09–11.')

    # CELL 09 — Report clean versus shuffled performance
    shuffled_logits = {}
    for name in DISPLAY_NAMES:
        shuffled_logits.update(load_result(name)['logits'])
    metric_rows, prediction_tables = [], []
    for model_name, logits in shuffled_logits.items():
        probabilities = logits.softmax(1)
        confidence, predictions = probabilities.max(1)
        clean = clean_predictions[model_name]
        clean_accuracy = (clean == y_test).float().mean().item()
        accuracy = (predictions == y_test).float().mean().item()
        metric_rows.append({
            'model': model_name, 'clean_accuracy': clean_accuracy,
            'shuffled_accuracy': accuracy,
            'accuracy_drop_pp': 100 * (clean_accuracy - accuracy),
            'prediction_consistency': (predictions == clean).float().mean().item(),
            'macro_f1': f1_score(y_test.numpy(), predictions.numpy(),
                                 labels=list(range(num_classes)), average='macro', zero_division=0),
            'mean_max_confidence': confidence.mean().item(),
        })
        table = manifest.copy()
        table['model'] = model_name
        table['predicted_label'] = predictions.numpy()
        table['predicted_class'] = [class_names[int(i)] for i in predictions]
        table['clean_predicted_label'] = clean.numpy()
        table['matches_clean'] = (predictions == clean).numpy()
        table['confidence'] = confidence.numpy()
        prediction_tables.append(table)
    patch_results = pd.DataFrame(metric_rows)
    patch_predictions = pd.concat(prediction_tables, ignore_index=True)
    assert len(patch_results) == 4 and len(patch_predictions) == 2000
    display(patch_results.style.format({
        'clean_accuracy':'{:.2%}', 'shuffled_accuracy':'{:.2%}', 'accuracy_drop_pp':'{:+.2f}',
        'prediction_consistency':'{:.2%}', 'macro_f1':'{:.4f}', 'mean_max_confidence':'{:.2%}'
    }))

    # CELL 10 — Plot accuracy and prediction consistency
    patch_figure, axes = plt.subplots(1, 2, figsize=(13, 5))
    x = np.arange(len(patch_results))
    axes[0].bar(x - 0.18, 100 * patch_results['clean_accuracy'], 0.36, label='Clean')
    axes[0].bar(x + 0.18, 100 * patch_results['shuffled_accuracy'], 0.36, label='Shuffled')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].legend()
    axes[1].bar(x, 100 * patch_results['prediction_consistency'])
    axes[1].set_ylabel('Prediction consistency (%)')
    for ax in axes:
        ax.set_xticks(x)
        ax.set_xticklabels(patch_results['model'], rotation=25, ha='right', fontsize=8)
        ax.set_ylim(0, 100)
        ax.grid(axis='y', alpha=0.2)
    patch_figure.suptitle('Patch shuffling: same pixels, disrupted spatial organization')
    plt.tight_layout()
    plt.show()

    # CELL 11 — Save metrics, predictions and plot
    patch_results.to_csv(OUTPUT_DIR / 'patch_metrics.csv', index=False)
    patch_predictions.to_csv(OUTPUT_DIR / 'patch_predictions.csv', index=False)
    manifest.to_csv(OUTPUT_DIR / 'test_manifest.csv', index=False)
    patch_figure.savefig(OUTPUT_DIR / 'patch_comparison.png', dpi=180, bbox_inches='tight')
    print('Saved to:', OUTPUT_DIR)
    print('Per-backbone .pt files already contain shuffled features, logits, IDs and permutations.')

